In [1]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
input_folder  = "/home/mojtaba.kananisarche/Zh/UTA/feature_vectors/mobilenetv2_topremoved_avgpooling/raw"
output_folder_pca_transformers = f"{os.path.split(input_folder)[0]}/pca_transformers"
output_folder_pca_transformed  = f"{os.path.split(input_folder)[0]}/pca_transformed"
output_folder_standard_scalers = f"{os.path.split(input_folder)[0]}/standard_scalers"
output_folder_standard_scaled  = f"{os.path.split(input_folder)[0]}/standard_scaled"

output_folders = [output_folder_pca_transformers, 
                  output_folder_pca_transformed, 
                  output_folder_standard_scalers, 
                  output_folder_standard_scaled]
for output_folder in output_folders:
    os.makedirs(output_folder, exist_ok=True)

csv_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.csv'))]
column_count = pd.read_csv(os.path.join(input_folder, csv_files[0])).shape[1]
n_components=0.95

print(len(csv_files))
print(column_count)

175
1280


In [3]:
csv_files_folds = [[], [], [], [], []]
for csv_file in csv_files:
    csv_files_folds[int(csv_file[0])-1].append(csv_file)
csv_files_folds

[['1_04_10_30fps_sq_cropped_mobilenetv2.csv',
  '1_12_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_07_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_03_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_02_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_07_10_30fps_sq_cropped_mobilenetv2.csv',
  '1_09_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_05_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_10_10_30fps_sq_cropped_mobilenetv2.csv',
  '1_10_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_10_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_02_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_05_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_02_10_30fps_sq_cropped_mobilenetv2.csv',
  '1_08_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_04_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_11_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_07_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_08_10_30fps_sq_cropped_mobilenetv2.csv',
  '1_06_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_08_5_30fps_sq_cropped_mobilenetv2.csv',
  '1_01_0_30fps_sq_cropped_mobilenetv2.csv',
  '1_

In [4]:
for i in range(len(csv_files_folds)):
    print(f"====={i+1}=====")
    
    # Define file paths
    pca_transformer_filename = os.path.join(output_folder_pca_transformers, f'scaler_pca_{i+1}.pkl')
    pca_transformed_df_path = os.path.join(output_folder_pca_transformed, f'pca_transformed_test_validation_{i+1}.csv')
    scaler_filename = os.path.join(output_folder_standard_scalers, f'scaler_{i+1}.pkl')
    scaled_df_path = os.path.join(output_folder_standard_scaled, f'scaled_test_validation_{i+1}.csv')
    
    # Check if all files exist
    if (os.path.exists(pca_transformer_filename) and os.path.exists(pca_transformed_df_path) and 
        os.path.exists(scaler_filename) and os.path.exists(scaled_df_path)):
        print("All required files already exist. Skipping iteration.")
        continue
    
    # If PCA-transformed CSV exists but standard scaler files don't, load PCA-transformed CSV and continue from there
    if os.path.exists(pca_transformed_df_path) and os.path.exists(pca_transformer_filename) and \
       (not os.path.exists(scaler_filename) or not os.path.exists(scaled_df_path)):
        print("PCA-transformed data found. Skipping PCA step and proceeding with scaling.")
        pca_transformed_df = pd.read_csv(pca_transformed_df_path)
        pca_transformed_df['fold'] = pca_transformed_df['fold'].astype(int)
        pca_transformed_df['subject_id'] = pca_transformed_df['subject_id'].astype(int)
        pca_transformed_df['target'] = pca_transformed_df['target'].astype(int)
        train = pca_transformed_df[pca_transformed_df['split'] == 'train'].reset_index(drop=True)
        test_validation = pca_transformed_df[pca_transformed_df['split'] == 'test_validation'].reset_index(drop=True)
        
        del pca_transformed_df
        gc.collect()
    
    else:
        data = np.empty((0, column_count))  # Initialize empty array with known column count
        
        print("\n***Train***")
        train = pd.DataFrame()
        for j in range(len(csv_files_folds)):
            if i == j: continue
            print(f"-----{j+1}-----")
            for csv_file in csv_files_folds[j]:
                temp_df = pd.read_csv(os.path.join(input_folder, csv_file))
                temp_df['filename'] = os.path.splitext(os.path.basename(csv_file))[0]
                temp_df['frame'] = temp_df.index
                temp_df['fold'] = temp_df['filename'].apply(lambda x: int(x.split('_')[0]))
                temp_df['subject_id'] = temp_df['filename'].apply(lambda x: int(x.split('_')[1]))
                temp_df['split'] = 'train'
                temp_df['target'] = temp_df['filename'].apply(lambda x: int(x.split('_')[2]))
                print(temp_df.shape)
                train = pd.concat([train, temp_df], axis=0, ignore_index=True)
        
        print("\n***Test_Validation***")
        test_validation = pd.DataFrame()
        for csv_file in csv_files_folds[i]:
            temp_df = pd.read_csv(os.path.join(input_folder, csv_file))
            temp_df['filename'] = os.path.splitext(os.path.basename(csv_file))[0]
            temp_df['frame'] = temp_df.index
            temp_df['fold'] = temp_df['filename'].apply(lambda x: int(x.split('_')[0]))
            temp_df['subject_id'] = temp_df['filename'].apply(lambda x: int(x.split('_')[1]))
            temp_df['split'] = 'test_validation'
            temp_df['target'] = temp_df['filename'].apply(lambda x: int(x.split('_')[2]))
            print(temp_df.shape)
            test_validation = pd.concat([test_validation, temp_df], axis=0, ignore_index=True)
        
        scaler_pca_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=n_components))
        ])
        
        pipeline_exclude_cols = ['filename', 'frame', 'fold', 'subject_id', 'split', 'target']
        pipeline_include_cols = train.columns.difference(pipeline_exclude_cols, sort=False)
        
        scaler_pca_pipeline.fit(train[pipeline_include_cols])
        with open(pca_transformer_filename, 'wb') as f:
            pickle.dump(scaler_pca_pipeline, f)
        print(f'Saved PCA model: {pca_transformer_filename}')
        
        pipelined_column_names = [fn.replace('a','_') for fn in list(scaler_pca_pipeline.get_feature_names_out())]
        train = pd.concat([pd.DataFrame(scaler_pca_pipeline.transform(train[pipeline_include_cols]), 
                                        columns=pipelined_column_names), 
                           train[pipeline_exclude_cols]], axis=1)
        test_validation = pd.concat([pd.DataFrame(scaler_pca_pipeline.transform(test_validation[pipeline_include_cols]), 
                                                  columns=pipelined_column_names), 
                                     test_validation[pipeline_exclude_cols]], axis=1)
        
        pca_transformed_df = pd.concat([train, test_validation], axis=0).reset_index(drop=True)
        pca_transformed_df.to_csv(pca_transformed_df_path, index=False)
        print(f'Saved PCA transformed df: {pca_transformed_df_path}')
        
        del pca_transformed_df
        gc.collect()
    
    scaler = StandardScaler()
    scaler_exclude_cols = ['filename', 'frame', 'fold', 'subject_id', 'split', 'target']
    scaler_include_cols = train.columns.difference(scaler_exclude_cols, sort=False)
    
    scaler.fit(train[scaler_include_cols])
    with open(scaler_filename, 'wb') as f:
        pickle.dump(scaler, f)
    print(f'Saved Scaler model: {scaler_filename}')
    
    train = pd.concat([pd.DataFrame(scaler.transform(train[scaler_include_cols]), 
                                    columns=scaler_include_cols), 
                       train[scaler_exclude_cols]], axis=1)
    test_validation = pd.concat([pd.DataFrame(scaler.transform(test_validation[scaler_include_cols]), 
                                              columns=scaler_include_cols), 
                                 test_validation[scaler_exclude_cols]], axis=1)
    
    scaled_df = pd.concat([train, test_validation], axis=0).reset_index(drop=True)
    scaled_df.to_csv(scaled_df_path, index=False)
    print(f'Saved scaled df: {scaled_df_path}')
    
    del scaled_df, test_validation, train
    gc.collect()


=====1=====

***Train***
-----2-----
(19842, 1286)
(18692, 1286)
(18386, 1286)
(18139, 1286)
(19186, 1286)
(18080, 1286)
(18617, 1286)
(18263, 1286)
(23159, 1286)
(22213, 1286)
(20622, 1286)
(18074, 1286)
(21895, 1286)
(14192, 1286)
(18105, 1286)
(18048, 1286)
(18170, 1286)
(18240, 1286)
(18050, 1286)
(18285, 1286)
(18083, 1286)
(18335, 1286)
(18054, 1286)
(18482, 1286)
(18360, 1286)
(17131, 1286)
(18184, 1286)
(18250, 1286)
(19301, 1286)
(13155, 1286)
(18541, 1286)
(18121, 1286)
(21561, 1286)
(19547, 1286)
(18109, 1286)
(18188, 1286)
-----3-----
(18508, 1286)
(14713, 1286)
(19052, 1286)
(19287, 1286)
(20867, 1286)
(20459, 1286)
(18280, 1286)
(5483, 1286)
(17647, 1286)
(26996, 1286)
(18045, 1286)
(22702, 1286)
(18676, 1286)
(26998, 1286)
(18829, 1286)
(19919, 1286)
(18894, 1286)
(18667, 1286)
(18135, 1286)
(20997, 1286)
(18040, 1286)
(18083, 1286)
(26441, 1286)
(12631, 1286)
(18196, 1286)
(20419, 1286)
(19367, 1286)
(16823, 1286)
(18388, 1286)
(18097, 1286)
(20980, 1286)
(18611, 1286)


KeyboardInterrupt: 